# Module 0 — The foundation network (bring-your-own VPC)

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## The question

> *"The Workshop 1 Neptune stack asks me for a `VpcId`, a list of `SubnetIds`, and a
> `VpcCidr`. The Workshop 2 CDK looks up a VPC and private subnets that already exist.
> Where do those come from — and what, exactly, do they have to look like?"*

Every later module assumes a network is already there. This module builds that network —
the minimal, correct foundation the rest of ATLAS stands on — as a single CloudFormation
template, and explains why each piece exists. It is **Module 0**: the step before Module 1.

## The concept — why this is a *bring-your-own* foundation

Be honest about what this module is and is not. ATLAS does **not** ship a bespoke,
hand-tuned production VPC. The foundation the workshop authors themselves run on today is
an existing Amazon SageMaker Studio VPC — a network that already happened to be there.
That is the normal case in a real institution: the network predates the project.

So Module 0 does not *templatize a specific ATLAS VPC* (there isn't one). It builds the
**equivalent minimal foundation** a customer needs *when they do not already have a
suitable VPC* — at a clean CIDR, with the one non-obvious correctness constraint baked in
(the AZ-exclusion rule, below). If you already have a VPC that meets the contract, you can
skip the deploy and just feed your existing values to Module 1. If you don't, this template
gives you one that is known to satisfy every downstream requirement.

The governing principle is **traceability**: every resource in the template maps to a
concrete requirement in Workshop 1 or Workshop 2. Nothing is there "because a VPC usually
has one." If a construct can't be traced to a consumer, it isn't in the template.

## What the stacks require — the contract, traced

The foundation's job is to satisfy two consumers. Here is each requirement and the line
of code that imposes it.

### Consumer 1 — Workshop 1's Neptune stack

[`../infrastructure/atlas-neptune-twotier.yaml`](../infrastructure/atlas-neptune-twotier.yaml)
declares these **parameters** — the foundation's outputs must supply them:

| WS1 parameter | Type | What the foundation supplies |
|---|---|---|
| `VpcId` | `AWS::EC2::VPC::Id` | the foundation VPC |
| `SubnetIds` | `List<AWS::EC2::Subnet::Id>` — *"At least two subnets in different AZs"* | the two private subnets |
| `VpcCidr` | `String` | the VPC CIDR (Neptune's SG ingress on 8182) |
| `ProjectTag` | `String` (default `atlas`) | same tag value, end to end |

That `SubnetIds` requirement is not cosmetic: the Neptune `DBSubnetGroup` in that template
*requires subnets in 2+ Availability Zones*. Two private subnets in two AZs is the minimum
that satisfies it. The Neptune two-tier design itself is taught in
[`03_two_tier_neptune.ipynb`](03_two_tier_neptune.ipynb).

### Consumer 2 — Workshop 2's CDK networking

[`../../use-case-applications/cdk/lib/constructs/networking.ts`](../../use-case-applications/cdk/lib/constructs/networking.ts)
does **not create** a VPC — it *looks one up* (`ec2.Vpc.fromLookup`) and imports the private
subnets, then attaches the security groups WS2 needs. Its inputs are `vpcId: string` and
`privateSubnetIds: string[]` — fed from the CDK context in
[`../../use-case-applications/cdk/cdk.json`](../../use-case-applications/cdk/cdk.json)
(the `vpcId` and `privateSubnetIds` keys, empty until you fill them from the foundation
outputs).

WS2 also needs **outbound HTTPS (443)**: `networking.ts` opens Lambda egress *"to Bedrock
and AWS APIs via NAT"* and ECS egress *"to ECR and S3."* Private subnets have no public
IP, so that egress must route through a **NAT gateway** — which in turn needs an **Internet
Gateway**, a **public subnet** to live in, an **Elastic IP**, and **route tables**. That is
the entire reason those resources exist in the template.

### The traceability table (every construct → its requirement)

| Construct in `atlas-foundation.yaml` | Required by |
|---|---|
| `Vpc` | WS1 `VpcId` param · WS2 `vpcId` lookup |
| `PrivateSubnet1`, `PrivateSubnet2` (2 AZs) | WS1 `SubnetIds` (Neptune DBSubnetGroup 2+ AZ) · WS2 `privateSubnetIds` (Lambda/ECS/AgentCore placement) |
| `PublicSubnet` | hosts the NAT gateway |
| `NatGateway` + `NatEip` | WS2 Lambda/ECS/agent 443 egress to Bedrock / S3 / ECR |
| `InternetGateway` + attachment | the NAT's path to the internet |
| route tables + routes + associations | private→NAT, public→IGW |

### What is deliberately ABSENT (and why)

Nothing in either workshop references them, so adding them would be untraceable boilerplate:

- **VPC endpoints** — the stacks reach Neptune *in-VPC* (port 8182, no endpoint needed) and
  reach Bedrock / S3 / ECR over HTTPS *via the NAT gateway*. No code references a VPC
  endpoint, so there is none.
- **Transit Gateway** — there is no second network to peer with.
- **VPC flow logs** — not referenced by any consumer; a real production hardening step, but
  not a workshop *requirement*. Add it in your own environment if your policy needs it.

## The AZ-exclusion lesson — the most valuable thing in Module 0

This is the one piece of knowledge that will save you a half-day of confusing debugging.

**The rule.** Amazon Bedrock AgentCore VPC mode — used by Workshop 2's agent runtimes —
can only place its network interfaces in a *subset* of Availability Zones. In `us-east-1`
the supported zones are the **AZ-IDs** `use1-az1`, `use1-az2`, and `use1-az4`. It does
**not** support `use1-az6`.

**Why AZ-IDs, not AZ names.** An AZ *name* like `us-east-1b` is **not** the same physical
zone in every account — AWS shuffles the name→zone mapping per account so load spreads
evenly. The stable, account-independent identifier is the **AZ-ID** (`use1-azN`). So the
unsupported zone is reliably `use1-az6`; *which name* that wears depends on the account.
Run the cell below to see the mapping in **this** account — `use1-az6` is the one to avoid.

**What goes wrong if you ignore it.** If a private subnet lands in `use1-az6`, everything
in Workshop 1 still works (Neptune doesn't care). Then, much later, Workshop 2's AgentCore
runtimes fail to place their ENIs — with an error that points at the runtime, not at the
subnet you chose three steps earlier. It is a classic far-downstream failure.

**How the template bakes it in.** `atlas-foundation.yaml` selects the private subnets by
`AvailabilityZoneId`, and constrains the parameter `AllowedValues` to `[use1-az1, use1-az2,
use1-az4]`. `use1-az6` is **not an allowed value**, so you *cannot* select it — the
mistake is structurally impossible, not merely discouraged. A `Rules` assertion also forces
the two private-subnet AZ-IDs to differ, guaranteeing the 2-AZ Neptune requirement.

This is the same rule that Workshop 2's
[`networking.ts`](../../use-case-applications/cdk/lib/constructs/networking.ts) encodes
(see its comment block: *"AgentCore supports use1-az1 / use1-az2 / use1-az4 … it does NOT
support use1-az6 … drop any subnet that lives in us-east-1b"*). Module 0 moves that rule
upstream, into the network's creation, so the bad subnet never exists in the first place.

In [ ]:
# Show THIS account's AZ name -> AZ-ID mapping, and flag the AgentCore-unsupported zone.
# (Read-only: describe-availability-zones creates nothing.)
import subprocess, json

SUPPORTED = {'use1-az1', 'use1-az2', 'use1-az4'}   # AgentCore VPC mode, us-east-1
UNSUPPORTED = {'use1-az6'}                          # NOT supported — must be excluded

raw = subprocess.run(
    ['aws', 'ec2', 'describe-availability-zones', '--region', 'us-east-1',
     '--query', 'AvailabilityZones[].{Name:ZoneName,Id:ZoneId}', '--output', 'json'],
    capture_output=True, text=True)
if raw.returncode != 0:
    print('describe-availability-zones unavailable in this environment:')
    print(raw.stderr.strip())
else:
    for z in sorted(json.loads(raw.stdout), key=lambda x: x['Id']):
        tag = 'SUPPORTED  ' if z['Id'] in SUPPORTED else ('EXCLUDE -> ' if z['Id'] in UNSUPPORTED else '(unused)   ')
        print(f"  {z['Id']:9} = {z['Name']:11} {tag}")
    print()
    print('Pick private-subnet AZ-IDs from the SUPPORTED set. The template enforces this')
    print('via AllowedValues, so use1-az6 cannot be chosen even by accident.')

## The template — and the proof it is structurally sound

The foundation lives in
[`../infrastructure/atlas-foundation.yaml`](../infrastructure/atlas-foundation.yaml), beside
Workshop 1's Neptune template. The cells below **inspect** it and **dry-validate** it. None
of this creates a live resource.

"Dry-validate" means three non-destructive checks: `cfn-lint` (structural lint),
`aws cloudformation validate-template` (the service's own parser), and a **change-set
preview** (CloudFormation computes *what it would create* without creating it — you then
delete the change set). Together they prove the template is **config-verified**: well-formed,
accepted by CloudFormation, and would add exactly the resources you intend. Read the honest
limit at the end of the notebook before calling it "done."

In [ ]:
# Inspect the template: its parameters, the resource inventory, and the outputs.
import os
TEMPLATE = '../infrastructure/atlas-foundation.yaml'
assert os.path.exists(TEMPLATE), f'template not found at {TEMPLATE}'

text = open(TEMPLATE).read()
# Resource logical-ids are top-level keys under Resources: (2-space indent, no deeper).
import re
res = re.findall(r'^  (\w+):\n    Type: (AWS::[\w:]+)', text, re.M)
print(f'{len(res)} resources declared:')
for logical, rtype in res:
    print(f'  {logical:38} {rtype}')

In [ ]:
# Dry-validation 1 + 2: cfn-lint (if installed) and the CloudFormation service validator.
def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p.returncode, (p.stdout or p.stderr).strip()

rc, out = run(['cfn-lint', TEMPLATE])
print('cfn-lint           :', 'CLEAN' if rc == 0 else f'issues (exit {rc})')
if out:
    print('  ', out.replace(chr(10), chr(10) + '   '))

rc, out = run(['aws', 'cloudformation', 'validate-template',
               '--template-body', f'file://{TEMPLATE}',
               '--query', 'Parameters[].ParameterKey', '--output', 'json'])
print('validate-template  :', 'VALID' if rc == 0 else 'ERROR')
print('  parameters:', out if rc == 0 else out)

### The change-set preview (the strongest non-live proof)

A **change set** asks CloudFormation: *"if I deployed this, what would you do?"* Created
with `--change-set-type CREATE` against a stack name that doesn't exist yet, it computes the
full resource list **without instantiating anything** (the placeholder stack sits in
`REVIEW_IN_PROGRESS`, which holds no real resources). You inspect the answer, then **delete**
the change set and the placeholder — leaving zero residue.

This was run while building the module. The change set previewed **15 resources**, every one
an `Add`: the `Vpc`; `PublicSubnet`, `PrivateSubnet1`, `PrivateSubnet2`; `InternetGateway` +
attachment; `NatGateway` + `NatEip`; the public and private route tables, their default
routes, and the three subnet associations. Then the change set and placeholder were deleted.

The cell below reproduces it end to end — create, list, delete — so you can confirm it
yourself. It is written to clean up after itself even if a step fails.

In [ ]:
# Change-set preview: create (no execute) -> list resources -> delete. Creates NO resources.
STACK = 'atlas-foundation-drycheck'
CS    = 'atlas-foundation-preview'

def cfn(args):
    return subprocess.run(['aws', 'cloudformation', *args], capture_output=True, text=True)

created = False
try:
    r = cfn(['create-change-set', '--stack-name', STACK, '--change-set-name', CS,
             '--change-set-type', 'CREATE', '--template-body', f'file://{TEMPLATE}',
             '--capabilities', 'CAPABILITY_NAMED_IAM', '--query', 'Id', '--output', 'text'])
    if r.returncode != 0:
        print('create-change-set failed (need cfn permissions?):'); print(' ', r.stderr.strip())
    else:
        created = True
        cfn(['wait', 'change-set-create-complete', '--stack-name', STACK, '--change-set-name', CS])
        r = cfn(['describe-change-set', '--stack-name', STACK, '--change-set-name', CS,
                 '--query', 'Changes[].ResourceChange.{A:Action,T:ResourceType,L:LogicalResourceId}',
                 '--output', 'json'])
        changes = json.loads(r.stdout or '[]')
        print(f'change set previews {len(changes)} resources (all should be "Add"):')
        for c in changes:
            print(f"  {c['A']:5} {c['L']:38} {c['T']}")
finally:
    # Always clean up — never leave a change set or placeholder behind.
    if created:
        cfn(['delete-change-set', '--stack-name', STACK, '--change-set-name', CS])
        cfn(['delete-stack', '--stack-name', STACK])
        print('\ncleaned up: change set + REVIEW_IN_PROGRESS placeholder deleted (no resources created).')

## The output-contract cross-check (does it actually fit the consumers?)

Structural soundness isn't enough — the outputs have to be the *exact* shape the consumers
expect. This is the cross-check that ties Module 0 to Module 1 and to Workshop 2.

| `atlas-foundation.yaml` output | → WS1 Neptune parameter | → WS2 CDK context | Fits? |
|---|---|---|---|
| `VpcId` | `VpcId` (`AWS::EC2::VPC::Id`) | `vpcId` | ✓ |
| `PrivateSubnetIds` (comma-joined) | `SubnetIds` (`List<AWS::EC2::Subnet::Id>`) | `privateSubnetIds` | ✓ |
| `VpcCidr` | `VpcCidr` (`String`) | — | ✓ |

`PrivateSubnetIds` is emitted as a comma-delimited string precisely because that is what a
`List<AWS::EC2::Subnet::Id>` parameter accepts directly, and what the CDK context array can
be split from. Two subnets in two supported AZs satisfies both Neptune's 2-AZ DBSubnetGroup
and AgentCore's placement. The cell below re-derives this check from the files themselves.

In [ ]:
# Re-derive the cross-check from the actual files (no hardcoding).
NEPTUNE = '../infrastructure/atlas-neptune-twotier.yaml'
NET     = '../../use-case-applications/cdk/lib/constructs/networking.ts'

foundation_outputs = set(re.findall(r'^  (\w+):\n    Description:', open(TEMPLATE).read().split('Outputs:',1)[1], re.M))
neptune_params = set(re.findall(r'^  (\w+):\n    Type:', open(NEPTUNE).read().split('Parameters:',1)[1].split('Resources:',1)[0], re.M))
net_inputs = set(re.findall(r'(\w+):\s', ''.join(l for l in open(NET) if 'vpcId' in l or 'privateSubnetIds' in l)))

print('foundation outputs :', sorted(foundation_outputs))
print('WS1 neptune params :', sorted(neptune_params))
print('WS2 networking ins :', sorted(i for i in net_inputs if i in {'vpcId','privateSubnetIds'}))
print()
# The three contract pairings:
checks = [
    ('VpcId',            'VpcId' in foundation_outputs and 'VpcId' in neptune_params),
    ('SubnetIds<-PrivateSubnetIds', 'PrivateSubnetIds' in foundation_outputs and 'SubnetIds' in neptune_params),
    ('VpcCidr',          'VpcCidr' in foundation_outputs and 'VpcCidr' in neptune_params),
    ('WS2 vpcId',         'vpcId' in net_inputs),
    ('WS2 privateSubnetIds', 'privateSubnetIds' in net_inputs),
]
for name, ok in checks:
    print(('OK  ' if ok else 'FAIL'), name)
assert all(ok for _, ok in checks), 'output contract does not satisfy a consumer'
print('\nOutput contract satisfied: the foundation feeds WS1 + WS2 exactly.')

## Deploy it, and wire the outputs into the rest of ATLAS

When you are ready to actually create the network (in *your* account, outside this dry-run),
the flow is:

```bash
# 1. Deploy the foundation (pick two SUPPORTED, distinct AZ-IDs; defaults are use1-az1/az2).
aws cloudformation deploy \
  --template-file agentic-semantic-layer/infrastructure/atlas-foundation.yaml \
  --stack-name atlas-foundation \
  --capabilities CAPABILITY_NAMED_IAM \
  --parameter-overrides VpcCidr=10.0.0.0/16 PrivateSubnet1AzId=use1-az1 PrivateSubnet2AzId=use1-az2

# 2. Capture the three outputs.
aws cloudformation describe-stacks --stack-name atlas-foundation \
  --query 'Stacks[0].Outputs' --output table
```

Then feed them forward:

- **Workshop 1** — pass `VpcId`, `PrivateSubnetIds` (as `SubnetIds`), and `VpcCidr` as the
  parameters when you deploy
  [`atlas-neptune-twotier.yaml`](../infrastructure/atlas-neptune-twotier.yaml) in
  [`03_two_tier_neptune.ipynb`](03_two_tier_neptune.ipynb).
- **Workshop 2** — put `VpcId` and `PrivateSubnetIds` into the `vpcId` and
  `privateSubnetIds` keys of
  [`../../use-case-applications/cdk/cdk.json`](../../use-case-applications/cdk/cdk.json)
  (they ship empty). The WS2 pre-flight
  [`00_preflight.ipynb`](../../use-case-applications/notebooks/phase-1-referral/00_preflight.ipynb)
  verifies the substrate, and the deploy is
  [`08_deploy.ipynb`](../../use-case-applications/notebooks/phase-1-referral/08_deploy.ipynb).
- **CDK bootstrap** — Workshop 2 is a CDK app, so the account/region must be bootstrapped
  once before `cdk deploy`:

```bash
cdk bootstrap aws://<account-id>/us-east-1
```

## The honest limit — dry-validated, not live-proven

Be precise about what this module has and has not demonstrated.

**What is proven (config-verified).** The template is `cfn-lint`-clean, accepted by
`aws cloudformation validate-template`, and a change-set preview confirmed it *would* create
exactly the 15 intended resources (then the change set was deleted — **no live resources
were created**). The outputs were cross-checked against the real consumer inputs and fit
exactly. The AZ-exclusion rule is enforced structurally via `AllowedValues`.

**What is NOT yet proven (live).** Nobody has run **WS0 → WS1 → WS2 from a genuinely empty
account** to confirm the whole chain stands up from nothing. That end-to-end, clean-account
proof is a separate exercise, and it is **tabled** — it needs a truly empty account (the
workshop's current accounts already carry a foundation, so they cannot prove "from
nothing"). Until then, treat Module 0 as *structurally sound and contract-matched*, not as
*live-proven*.

This honesty is the point of the workshop, not a footnote: a config-verified template is a
strong, useful artifact — and calling it that, rather than "it works," is exactly the
discipline ATLAS teaches everywhere else.

## What just changed

You now have the network the rest of ATLAS assumed into existence: a single, traceable
CloudFormation template that produces a VPC, two private subnets in AgentCore-supported
AZs (with `us-east-1b` / `use1-az6` made structurally unselectable), and the NAT path that
lets in-VPC workloads reach Bedrock, S3, and ECR. Its three outputs — `VpcId`,
`PrivateSubnetIds`, `VpcCidr` — are the exact inputs Workshop 1's Neptune stack and
Workshop 2's CDK consume. The template is dry-validated (config-verified), and you know the
one rule that would otherwise have cost you a half-day. The next module turns this network
into a place to run a graph: [`03_two_tier_neptune.ipynb`](03_two_tier_neptune.ipynb).